# Solar filament segmentation: full pipeline

This notebook runs the whole pipeline of this repository, from the competition images to `submission.csv`:

1. **Data**: MAGFiLO images and annotations from the Kaggle competition *Solar Filament Segmentation Challenge 2026*.
2. **Preprocessing**: limb darkening and local background divided out (`preprocess.py`).
3. **Model**: U-Net with an ImageNet-pretrained ResNet-34 encoder and a full-resolution skip (`resnet_unet.py`), trained at 2048x2048 on 768x768 crops with soft labels (fraction of annotators marking each pixel) and BCE + Dice + clDice + Lovasz loss (`main.py`, `train_class.py`, `losses.py`).
4. **Inference**: overlapping 768x768 tiles blended into one probability map (`tiling.py`).
5. **Post-processing**: threshold, direction-aware grouping of pieces (`grouping.py`), minimum size and confidence.
6. **Submission**: one row per filament, `filament_id` = `<image name>_<n>`, `segmentation_rle` = pycocotools RLE counts.

By default it uses the trained weights included in the repository (`deliverables/weights/resnet34_unet_2048.pt`, 49 MB),
so the only extra input is the competition data. Set `RUN_TRAINING = True` to retrain from scratch first
(~11 h on an Apple M1, MPS) and `RUN_VALIDATION = True` to recompute the validation PQ (~30 min on an M1).

**Setup** (from the repository folder):
```
pip install -r requirements.txt
```
Put the Kaggle competition files in `MAGFiLO_1.0_Kaggle_2026/` (as downloaded: `train/train_images/`,
`train/MAGFiLO_1.0_Annotations_kaggle2026_train.json`, `test/test_images/`), then run all cells.

In [ ]:
# ---- settings
RUN_TRAINING = False     # True: train a new model (main.py) and use it instead of the included weights
RUN_VALIDATION = False   # True: score the model on the validation split (PQ, as reported in README.md)
WEIGHTS = "deliverables/weights/resnet34_unet_2048.pt"
OUT = "submission.csv"
LIMIT = None             # e.g. 3 for a quick check on the first 3 test images

import os, sys, glob, json, csv, time, subprocess
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   # torch and OpenCV each ship an OpenMP runtime on macOS
sys.path[:0] = [os.path.abspath("."), os.path.abspath("deliverables")]   # run from the repository folder
import numpy as np, cv2, torch
import matplotlib.pyplot as plt
from pycocotools import mask as mask_utils

import preprocess, tiling, evaluate, predict     # project modules
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
DATA = "MAGFiLO_1.0_Kaggle_2026"
for path in (evaluate.IMAGE_DIR, evaluate.ANNOTATION_FILE, predict.TEST_IMAGES):
    assert os.path.exists(path), f"missing {path}: put the Kaggle competition files in {DATA}/"
print("device:", device, "| torch", torch.__version__)

## 1. Data

707 training images (2048x2048 grayscale JPEG from the GONG network) with 1,154 annotation sets: 296 images were
annotated by 2 or 3 people independently, and every copy is a complete annotation. 180 unlabelled test images.

In [ ]:
ann = json.load(open(evaluate.ANNOTATION_FILE))
files = {i["file_name"] for i in ann["images"]}
test_paths = sorted(glob.glob(os.path.join(predict.TEST_IMAGES, "*.jpeg")))
print(f"training images: {len(files)}, annotation sets: {len(ann['images'])}, filaments: {len(ann['annotations'])}")
print(f"test images: {len(test_paths)}")

## 2. Preprocessing

The disk is darker towards the edge (limb darkening), and the background brightness varies across the disk and
between images. `preprocess.local_normalise` fits the solar disk, divides out the radial brightness profile and a
local median background, and standardises each image (quiet disk ~0, filaments strongly negative, sky = -5).

In [ ]:
img = cv2.imread(test_paths[0], cv2.IMREAD_GRAYSCALE)
norm = preprocess.local_normalise(img)
fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(img, cmap="gray"); ax[0].set_title("Raw H-alpha image")
ax[1].imshow(norm, cmap="gray", vmin=-5, vmax=3); ax[1].set_title("After local_normalise")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## 3. Training (optional)

Training settings live in the `config` dictionary of `main.py` (logged to MLflow with every run) and are saved in
every checkpoint. `main.py --config-from <weights>` starts a new run with the settings stored in a weights file, so
this cell retrains the included model's configuration exactly, whatever `main.py` currently holds for newer
experiments. Training saves the best checkpoint (by validation IoU) in `checkpoints/`;
`deliverables/export_weights.py` keeps only the model weights in half precision. Skipped by default: the included
weights are the exported best model (run `2026-11-14_03-27_5981095`, validation IoU 0.546).

In [ ]:
if RUN_TRAINING:
    # a new run with exactly the settings the included model was trained with (stored in the weights file)
    subprocess.run([sys.executable, "main.py", "--config-from", WEIGHTS], check=True)
    newest = max(glob.glob("checkpoints/*.pt"), key=os.path.getmtime)
    newest = newest.replace("_last.pt", ".pt")       # the best checkpoint, not the resume file
    WEIGHTS = "deliverables/weights/retrained.pt"
    subprocess.run([sys.executable, "deliverables/export_weights.py", newest, WEIGHTS], check=True)
model, config = evaluate.load_model(WEIGHTS, device)
print(f"{WEIGHTS}: model {config['model']}, {sum(p.numel() for p in model.parameters())/1e6:.1f}M parameters, "
      f"resolution {config['resize_to']}, tiles {config['tile_size']} (overlap {config['tile_overlap']}), "
      f"preprocessing {config.get('preprocess')}")

## 4. Validation (optional)

`evaluate.py` scores the model on the validation split (20% of the training image files, seed 42, grouped by
file) with the competition's Panoptic Quality, against every annotator copy. With the post-processing below it
reports **PQ 0.440** (SQ 0.678, RQ 0.649; 95% bootstrap CI 0.414-0.462); annotators scored against each other reach 0.345.

In [ ]:
if RUN_VALIDATION:
    subprocess.run([sys.executable, "evaluate.py", "--checkpoint", WEIGHTS,
                    "--threshold", str(predict.THRESHOLD), "--close", str(predict.CLOSE),
                    "--min-area", str(predict.MIN_AREA), "--min-conf", str(predict.MIN_CONF)], check=True)

## 5. Inference and post-processing

For each test image: preprocessing as in training, the model on overlapping 768x768 tiles blended into one
2048x2048 probability map, then `predict.filaments`: threshold 0.5, pieces joined into one filament when their ends
face each other across a gap the model partly saw (`grouping.py`), filaments with mean
probability below 0.72 removed.

In [ ]:
print(f"threshold {predict.THRESHOLD}, grouping {predict.CLOSE}, min area {predict.MIN_AREA} px, min confidence {predict.MIN_CONF}")
paths = test_paths[:LIMIT]
rows, shown = [], {}
start = time.perf_counter()
for i, path in enumerate(paths, start=1):
    x, (w, h) = predict.load_image(path, config["resize_to"], config.get("preprocess"))
    with torch.no_grad():
        prob = tiling.predict_tiled(model, x.unsqueeze(0).to(device), config["tile_size"], config["tile_overlap"])[0, 0].float().cpu().numpy()
    if prob.shape != (h, w):
        prob = cv2.resize(prob, (w, h), interpolation=cv2.INTER_LINEAR)
    name = os.path.splitext(os.path.basename(path))[0]
    masks = predict.filaments(prob, predict.THRESHOLD, predict.CLOSE, predict.MIN_AREA, predict.MIN_CONF)
    rows += [(f"{name}_{n}", predict.rle_counts(m)) for n, m in enumerate(masks, start=1)]
    if len(shown) < 2 and len(masks) >= 3:
        shown[path] = masks
    if i % 20 == 0 or i == len(paths):
        print(f"  {i}/{len(paths)} images, {len(rows)} filaments, {time.perf_counter()-start:.0f}s")
with open(OUT, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["filament_id", "segmentation_rle"])
    writer.writerows(rows)
print(f"wrote {len(rows)} filaments ({len(rows)/len(paths):.1f} per image) to {OUT}")

## 6. Submission checks

Every row decodes to a 2048x2048 mask, IDs are unique and numbered per image, and the masks of one image do not overlap.

In [ ]:
with open(OUT, newline="") as f:
    reader = csv.reader(f)
    assert next(reader) == ["filament_id", "segmentation_rle"]
    sub = list(reader)
ids = [fid for fid, _ in sub]
assert len(ids) == len(set(ids)), "duplicate filament_id"
per_image = {}
for fid, counts in sub:
    stem, n = fid.rsplit("_", 1)
    m = mask_utils.decode({"size": [2048, 2048], "counts": counts.encode()})
    assert m.shape == (2048, 2048) and m.any()
    per_image.setdefault(stem, []).append((int(n), m))
for stem, items in per_image.items():
    assert sorted(n for n, _ in items) == list(range(1, len(items) + 1))
    assert sum(m.astype(np.int32) for _, m in items).max() <= 1, f"overlapping masks in {stem}"
print(f"OK: {len(sub)} filaments in {len(per_image)} of {len(paths)} images; "
      f"{len(paths) - len(per_image)} images without a predicted filament")

## 7. Example predictions on test images

In [ ]:
fig, axes = plt.subplots(1, max(len(shown), 1), figsize=(8 * max(len(shown), 1), 8), squeeze=False)
for ax, (path, masks) in zip(axes[0], shown.items()):
    ax.imshow(cv2.imread(path, cv2.IMREAD_GRAYSCALE), cmap="gray")
    for m in masks:
        contours, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        for c in contours:
            ax.plot(c[:, 0, 0], c[:, 0, 1], color="#eb6834", lw=1)
    ax.set_title(f"{os.path.basename(path)}: {len(masks)} predicted filaments"); ax.axis("off")
plt.tight_layout(); plt.show()